# **Support, Confidence, Lift, and Association Rules**

## **Libraries**

In [21]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "src" / "config.py").is_file():
    project_root = project_root / "Corte_3" / "PROYECTO_FINAL"
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.association_rules import AssociationData, AprioriRules

## **1. Question of Interest**

Which products tend to be purchased by the same customers, and what associations can Apriori method identify through support, confidence, and lift?

## **2. Database Loading and Creation of Customer-Product Matrix**

Load the processed dataset and create a matrix where rows represent customers and columns represent products.

In [22]:
data = AssociationData()
data.frame.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Name,Customer City,Customer Country,...,Order Item Quantity,Sales,Order Region,Order State,Order Status,Product Card Id,Product Name,Product Price,Shipping Mode,Order Month
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,...,1,327.75,Southeast Asia,Java Occidental,COMPLETE,1360,Smart watch,327.75,Standard Class,1
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,Sporting Goods,Caguas,Puerto Rico,...,1,327.75,South Asia,Rajastán,PENDING,1360,Smart watch,327.75,Standard Class,1
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,Sporting Goods,San Jose,EE. UU.,...,1,327.75,South Asia,Rajastán,CLOSED,1360,Smart watch,327.75,Standard Class,1
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,Sporting Goods,Los Angeles,EE. UU.,...,1,327.75,Oceania,Queensland,COMPLETE,1360,Smart watch,327.75,Standard Class,1
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,...,1,327.75,Oceania,Queensland,PENDING_PAYMENT,1360,Smart watch,327.75,Standard Class,1


In [23]:
data.frame.shape

(180519, 33)

### **2.1 Customer-Product Matrix**

Each value is the total quantity purchased by a customer for a product.

In [24]:
data.quantity_preview()

Product Card Id,19,24,35,37,44,58,60,61,78,93
Customer Id,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0
9,0,0,0,5,0,0,0,0,0,0


### **2.2 Binary Matrix**

For association rules, convert quantities to purchase presence: `True` means the customer purchased the product; `False` means no purchase is recorded.

Each transaction represents a customer's purchase history across the dataset, rather than a single order.

In [25]:
data.binary_preview()

Product Card Id,19,24,35,37,44,58,60,61,78,93
Customer Id,,,,,,,,,,
1,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False
5,False,False,False,False,False,False,False,False,False,False
6,False,False,False,False,False,False,False,False,False,False
7,False,False,False,False,False,False,False,False,False,False
8,False,False,False,False,False,False,False,False,False,False
9,False,False,False,True,False,False,False,False,False,False


In [26]:
data.describe()

Customers: 20652
Products: 118


## **3. Apriori**

### **3.1 Stablishing a Restricted Support**

In [27]:
data.describe_support()

count    118.000000
mean       0.041526
std        0.109291
min        0.000387
25%        0.003147
50%        0.013558
75%        0.014902
max        0.486442
dtype: float64

Products with support >= 1%:
72

Products with support >= 5%:
9


**Observations**
- We will use 5% as support

#### **3.2 Running Apriori**

In [28]:
apriori_analysis = AprioriRules(data, min_support=0.05, min_confidence=0.50)
apriori_analysis = apriori_analysis.fit_itemsets()

Frequent itemsets: 511

Itemset sizes:
itemsets
1      9
2     36
3     84
4    126
5    126
6     84
7     36
8      9
9      1
Name: count, dtype: int64


In [29]:
apriori_analysis.itemsets_of_size(9)

,support,itemsets,size
510,0.061786,"frozenset({1004, 365, 1073, 403, 627, 1014, 50...",9


#### **3.3 Defining Association Rules**

In [30]:
apriori_analysis = apriori_analysis.generate_rules()

Number of rules: 9206


In [34]:
rules_summary = apriori_analysis.summarize()

,antecedents,consequents,support,confidence,lift
9205,"frozenset({1073, 627, 957, 191})","frozenset({1004, 365, 403, 1014, 502})",0.061786,0.544369,2.598775
9183,"frozenset({1004, 627, 1014, 502, 957})","frozenset({1073, 403, 365, 191})",0.061786,0.500000,2.590567
9172,"frozenset({1004, 365, 1073, 627, 191})","frozenset({403, 957, 1014, 502})",0.061786,0.533222,2.573522
9204,"frozenset({627, 1004, 957, 191})","frozenset({365, 1073, 403, 1014, 502})",0.061786,0.511628,2.567713
9188,"frozenset({365, 1073, 627, 502, 191})","frozenset({403, 1004, 957, 1014})",0.061786,0.502758,2.567497
9203,"frozenset({627, 1014, 502, 957, 191})","frozenset({1073, 403, 1004, 365})",0.061786,0.576854,2.557574
9193,"frozenset({1073, 627, 403, 1014, 957})","frozenset({1004, 365, 502, 191})",0.061786,0.508164,2.553432
9192,"frozenset({365, 627, 502, 957, 191})","frozenset({1073, 403, 1004, 1014})",0.061786,0.533445,2.551935
9195,"frozenset({1073, 627, 403, 502, 191})","frozenset({1004, 365, 1014, 957})",0.061786,0.511423,2.551185
9156,"frozenset({365, 1073, 403, 1014, 957, 191})","frozenset({627, 1004, 502})",0.061786,0.523380,2.548654


In [35]:
positive_rules = apriori_analysis.positive()

Positive association rules: 9206


,antecedents,consequents,support,confidence,lift
9205,"frozenset({1073, 627, 957, 191})","frozenset({1004, 365, 403, 1014, 502})",0.061786,0.544369,2.598775
9183,"frozenset({1004, 627, 1014, 502, 957})","frozenset({1073, 403, 365, 191})",0.061786,0.500000,2.590567
9172,"frozenset({1004, 365, 1073, 627, 191})","frozenset({403, 957, 1014, 502})",0.061786,0.533222,2.573522
9204,"frozenset({627, 1004, 957, 191})","frozenset({365, 1073, 403, 1014, 502})",0.061786,0.511628,2.567713
9188,"frozenset({365, 1073, 627, 502, 191})","frozenset({403, 1004, 957, 1014})",0.061786,0.502758,2.567497
9203,"frozenset({627, 1014, 502, 957, 191})","frozenset({1073, 403, 1004, 365})",0.061786,0.576854,2.557574
9193,"frozenset({1073, 627, 403, 1014, 957})","frozenset({1004, 365, 502, 191})",0.061786,0.508164,2.553432
9192,"frozenset({365, 627, 502, 957, 191})","frozenset({1073, 403, 1004, 1014})",0.061786,0.533445,2.551935
9195,"frozenset({1073, 627, 403, 502, 191})","frozenset({1004, 365, 1014, 957})",0.061786,0.511423,2.551185
9156,"frozenset({365, 1073, 403, 1014, 957, 191})","frozenset({627, 1004, 502})",0.061786,0.523380,2.548654


**Observations**
- There are a big quantity of asociation rules, for simplicity, we will only analyse the top 10 in lift.

#### **3.4 Interpretation of the Top 10 of Association Rules**


Let $A \Rightarrow B$ represent an association rule, where $A$ is the antecedent and $B$ is the consequent.

| Metric | Formula | Interpretation |
|---|---|---|
| **Support** | $P(A \cap B)$ | Proportion of all customers who purchased both A and B. Measures how frequent the combination is. |
| **Confidence** | $P(B \mid A) = \frac{P(A \cap B)}{P(A)}$ | Proportion of customers who purchased A who also purchased B. Measures the conditional frequency of B given A. |
| **Lift** | $\frac{P(B \mid A)}{P(B)}$ | Measures how much more or less prevalent B is among customers who purchased A compared with all customers. |

A lift greater than 1 indicates a positive association; a lift equal to 1 indicates independence; and a lift below 1 indicates a negative association.

In [36]:
readable_rules = apriori_analysis.readable()

,antecedents,consequents,support,confidence,lift
9205,"191, 627, 957, 1073","365, 403, 502, 1004, 1014",0.061786,0.544369,2.598775
9183,"502, 627, 957, 1004, 1014","191, 365, 403, 1073",0.061786,0.500000,2.590567
9172,"191, 365, 627, 1004, 1073","403, 502, 957, 1014",0.061786,0.533222,2.573522
9204,"191, 627, 957, 1004","365, 403, 502, 1014, 1073",0.061786,0.511628,2.567713
9188,"191, 365, 502, 627, 1073","403, 957, 1004, 1014",0.061786,0.502758,2.567497
9203,"191, 502, 627, 957, 1014","365, 403, 1004, 1073",0.061786,0.576854,2.557574
9193,"403, 627, 957, 1014, 1073","191, 365, 502, 1004",0.061786,0.508164,2.553432
9192,"191, 365, 502, 627, 957","403, 1004, 1014, 1073",0.061786,0.533445,2.551935
9195,"191, 403, 502, 627, 1073","365, 957, 1004, 1014",0.061786,0.511423,2.551185
9156,"191, 365, 403, 957, 1014, 1073","502, 627, 1004",0.061786,0.523380,2.548654


In [38]:
readable_rules = apriori_analysis.readable(use_names=True)

,antecedents,consequents,support,confidence,lift
9205,"Nike Men's Free 5.0+ Running Shoe, Under Armour Girls' Toddler Spine Surge Runni, Diamondback Women's Serene Classic Comfort Bi, Pelican Sunstream 100 Kayak","Perfect Fitness Perfect Rip Deck, Nike Men's CJ Elite 2 TD Football Cleat, Nike Men's Dri-FIT Victory Golf Polo, Field & Stream Sportsman 16 Gun Fire Safe, O'Brien Men's Neoprene Life Vest",0.061786,0.544369,2.598775
9183,"Nike Men's Dri-FIT Victory Golf Polo, Under Armour Girls' Toddler Spine Surge Runni, Diamondback Women's Serene Classic Comfort Bi, Field & Stream Sportsman 16 Gun Fire Safe, O'Brien Men's Neoprene Life Vest","Nike Men's Free 5.0+ Running Shoe, Perfect Fitness Perfect Rip Deck, Nike Men's CJ Elite 2 TD Football Cleat, Pelican Sunstream 100 Kayak",0.061786,0.500000,2.590567
9172,"Nike Men's Free 5.0+ Running Shoe, Perfect Fitness Perfect Rip Deck, Under Armour Girls' Toddler Spine Surge Runni, Field & Stream Sportsman 16 Gun Fire Safe, Pelican Sunstream 100 Kayak","Nike Men's CJ Elite 2 TD Football Cleat, Nike Men's Dri-FIT Victory Golf Polo, Diamondback Women's Serene Classic Comfort Bi, O'Brien Men's Neoprene Life Vest",0.061786,0.533222,2.573522
9204,"Nike Men's Free 5.0+ Running Shoe, Under Armour Girls' Toddler Spine Surge Runni, Diamondback Women's Serene Classic Comfort Bi, Field & Stream Sportsman 16 Gun Fire Safe","Perfect Fitness Perfect Rip Deck, Nike Men's CJ Elite 2 TD Football Cleat, Nike Men's Dri-FIT Victory Golf Polo, O'Brien Men's Neoprene Life Vest, Pelican Sunstream 100 Kayak",0.061786,0.511628,2.567713
9188,"Nike Men's Free 5.0+ Running Shoe, Perfect Fitness Perfect Rip Deck, Nike Men's Dri-FIT Victory Golf Polo, Under Armour Girls' Toddler Spine Surge Runni, Pelican Sunstream 100 Kayak","Nike Men's CJ Elite 2 TD Football Cleat, Diamondback Women's Serene Classic Comfort Bi, Field & Stream Sportsman 16 Gun Fire Safe, O'Brien Men's Neoprene Life Vest",0.061786,0.502758,2.567497
9203,"Nike Men's Free 5.0+ Running Shoe, Nike Men's Dri-FIT Victory Golf Polo, Under Armour Girls' Toddler Spine Surge Runni, Diamondback Women's Serene Classic Comfort Bi, O'Brien Men's Neoprene Life Vest","Perfect Fitness Perfect Rip Deck, Nike Men's CJ Elite 2 TD Football Cleat, Field & Stream Sportsman 16 Gun Fire Safe, Pelican Sunstream 100 Kayak",0.061786,0.576854,2.557574
9193,"Nike Men's CJ Elite 2 TD Football Cleat, Under Armour Girls' Toddler Spine Surge Runni, Diamondback Women's Serene Classic Comfort Bi, O'Brien Men's Neoprene Life Vest, Pelican Sunstream 100 Kayak","Nike Men's Free 5.0+ Running Shoe, Perfect Fitness Perfect Rip Deck, Nike Men's Dri-FIT Victory Golf Polo, Field & Stream Sportsman 16 Gun Fire Safe",0.061786,0.508164,2.553432
9192,"Nike Men's Free 5.0+ Running Shoe, Perfect Fitness Perfect Rip Deck, Nike Men's Dri-FIT Victory Golf Polo, Under Armour Girls' Toddler Spine Surge Runni, Diamondback Women's Serene Classic Comfort Bi","Nike Men's CJ Elite 2 TD Football Cleat, Field & Stream Sportsman 16 Gun Fire Safe, O'Brien Men's Neoprene Life Vest, Pelican Sunstream 100 Kayak",0.061786,0.533445,2.551935
9195,"Nike Men's Free 5.0+ Running Shoe, Nike Men's CJ Elite 2 TD Football Cleat, Nike Men's Dri-FIT Victory Golf Polo, Under Armour Girls' Toddler Spine Surge Runni, Pelican Sunstream 100 Kayak","Perfect Fitness Perfect Rip Deck, Diamondback Women's Serene Classic Comfort Bi, Field & Stream Sportsman 16 Gun Fire Safe, O'Brien Men's Neoprene Life Vest",0.061786,0.511423,2.551185
9156,"Nike Men's Free 5.0+ Running Shoe, Perfect Fitness Perfect Rip Deck, Nike Men's CJ Elite 2 TD Football Cleat, Diamondback Women's Serene Classic Comfort Bi, O'Brien Men's Neoprene Life Vest, Pelican Sunstream 100 Kayak","Nike Men's Dri-FIT Victory Golf Polo, Under Armour Girls' Toddler Spine Surge Runni, Field & Stream Sportsman 16 Gun Fire Safe",0.061786,0.523380,2.548654


**Observations**

**Rule 1 — A → B**

- **A:** Nike Free running shoes; Under Armour toddler running shoes; Diamondback Serene bicycle; Pelican kayak.
- **B:** Perfect Fitness Rip Deck; Nike football cleats; Nike golf polo; Field & Stream gun safe; O’Brien life vest.
- **Support = 6.18%:** 6.18% of all customers purchased every product in both A and B.
- **Confidence = 54.44%:** Among customers who purchased every product in A, 54.44% also purchased every product in B.
- **Lift = 2.60:** Purchasing the entire B set was 2.60 times as prevalent among customers who purchased A as among customers overall.

**Rule 2 — A → B**

- **A:** Nike golf polo; Under Armour toddler running shoes; Diamondback Serene bicycle; Field & Stream gun safe; O’Brien life vest.
- **B:** Nike Free running shoes; Perfect Fitness Rip Deck; Nike football cleats; Pelican kayak.
- **Support = 6.18%:** 6.18% of all customers purchased every product in A and B.
- **Confidence = 50.00%:** Half of the customers who purchased all of A also purchased all of B.
- **Lift = 2.59:** Purchasing the entire B set was 2.59 times as prevalent among A purchasers as among customers overall.

**Rule 3 — A → B**

- **A:** Nike Free running shoes; Perfect Fitness Rip Deck; Under Armour toddler running shoes; Field & Stream gun safe; Pelican kayak.
- **B:** Nike football cleats; Nike golf polo; Diamondback Serene bicycle; O’Brien life vest.
- **Support = 6.18%:** 6.18% of all customers purchased every product in A and B.
- **Confidence = 53.32%:** Among customers who purchased all of A, 53.32% also purchased all of B.
- **Lift = 2.57:** Purchasing the entire B set was 2.57 times as prevalent among A purchasers as among customers overall.

**Rule 4 — A → B**

- **A:** Nike Free running shoes; Under Armour toddler running shoes; Diamondback Serene bicycle; Field & Stream gun safe.
- **B:** Perfect Fitness Rip Deck; Nike football cleats; Nike golf polo; O’Brien life vest; Pelican kayak.
- **Support = 6.18%:** 6.18% of all customers purchased every product in A and B.
- **Confidence = 51.16%:** Among customers who purchased all of A, 51.16% also purchased all of B.
- **Lift = 2.57:** Purchasing the entire B set was 2.57 times as prevalent among A purchasers as among customers overall.

**Rule 5 — A → B**

- **A:** Nike Free running shoes; Perfect Fitness Rip Deck; Nike golf polo; Under Armour toddler running shoes; Pelican kayak.
- **B:** Nike football cleats; Diamondback Serene bicycle; Field & Stream gun safe; O’Brien life vest.
- **Support = 6.18%:** 6.18% of all customers purchased every product in A and B.
- **Confidence = 50.28%:** Among customers who purchased all of A, 50.28% also purchased all of B.
- **Lift = 2.57:** Purchasing the entire B set was 2.57 times as prevalent among A purchasers as among customers overall.

**Rule 6 — A → B**

- **A:** Nike Free running shoes; Nike golf polo; Under Armour toddler running shoes; Diamondback Serene bicycle; O’Brien life vest.
- **B:** Perfect Fitness Rip Deck; Nike football cleats; Field & Stream gun safe; Pelican kayak.
- **Support = 6.18%:** 6.18% of all customers purchased every product in A and B.
- **Confidence = 57.69%:** Among customers who purchased all of A, 57.69% also purchased all of B.
- **Lift = 2.56:** Purchasing the entire B set was 2.56 times as prevalent among A purchasers as among customers overall.

**Rule 7 — A → B**

- **A:** Nike football cleats; Under Armour toddler running shoes; Diamondback Serene bicycle; O’Brien life vest; Pelican kayak.
- **B:** Nike Free running shoes; Perfect Fitness Rip Deck; Nike golf polo; Field & Stream gun safe.
- **Support = 6.18%:** 6.18% of all customers purchased every product in A and B.
- **Confidence = 50.82%:** Among customers who purchased all of A, 50.82% also purchased all of B.
- **Lift = 2.55:** Purchasing the entire B set was 2.55 times as prevalent among A purchasers as among customers overall.

**Rule 8 — A → B**

- **A:** Nike Free running shoes; Perfect Fitness Rip Deck; Nike golf polo; Under Armour toddler running shoes; Diamondback Serene bicycle.
- **B:** Nike football cleats; Field & Stream gun safe; O’Brien life vest; Pelican kayak.
- **Support = 6.18%:** 6.18% of all customers purchased every product in A and B.
- **Confidence = 53.34%:** Among customers who purchased all of A, 53.34% also purchased all of B.
- **Lift = 2.55:** Purchasing the entire B set was 2.55 times as prevalent among A purchasers as among customers overall.

**Rule 9 — A → B**

- **A:** Nike Free running shoes; Nike football cleats; Nike golf polo; Under Armour toddler running shoes; Pelican kayak.
- **B:** Perfect Fitness Rip Deck; Diamondback Serene bicycle; Field & Stream gun safe; O’Brien life vest.
- **Support = 6.18%:** 6.18% of all customers purchased every product in A and B.
- **Confidence = 51.14%:** Among customers who purchased all of A, 51.14% also purchased all of B.
- **Lift = 2.55:** Purchasing the entire B set was 2.55 times as prevalent among A purchasers as among customers overall.

**Rule 10 — A → B**

- **A:** Nike Free running shoes; Perfect Fitness Rip Deck; Nike football cleats; Diamondback Serene bicycle; O’Brien life vest; Pelican kayak.
- **B:** Nike golf polo; Under Armour toddler running shoes; Field & Stream gun safe.
- **Support = 6.18%:** 6.18% of all customers purchased every product in A and B.
- **Confidence = 52.34%:** Among customers who purchased all of A, 52.34% also purchased all of B.
- **Lift = 2.55:** Purchasing the entire B set was 2.55 times as prevalent among A purchasers as among customers overall.